# รัน `interleaved-reasoning` บน Kaggle

Notebook นี้**ต่อการทดลองจากที่ค้างใน repo** แล้วรันเมทริกซ์ที่เหลือจนจบ (resume ได้เสมอ)

> ⚠️ **repo เป็น private** → ต้องมี GitHub token (PAT) **ตั้งแต่เซลล์แรกที่ clone** และต้อง**เปิด Internet** ใน notebook settings

**เตรียม Kaggle (ครั้งเดียว):**

1. ดาวน์โหลดไฟล์ `kaggle.ipynb` จาก repo นี้ → kaggle.com → **Create → New Notebook → File → Import Notebook**
2. **Settings (แถบขวา) → Internet → Turn ON** (ต้องยืนยันเบอร์โทรไว้กับบัญชี Kaggle) — ปิดไว้จะ clone ไม่ได้
3. Accelerator = **None** (CPU 4 คอร์ฟรี — โค้ดล็อก CPU เพื่อ reproducibility, ไม่ต้องเปลือง GPU quota 30 ชม./สัปดาห์)
4. (แนะนำ) token: **Add-ons → Secrets → Add a new secret** → label **`GITHUB_TOKEN`** + กด **Attach** — หรือพิมพ์ในเซลล์ TOKEN ก็ได้
   PAT สร้างที่ github.com → Settings → Developer settings → Personal access tokens (fine-grained: เลือก repo นี้ + **Contents: Read/Write** | classic: scope `repo`)

**ลำดับการรัน (Run All ได้เลย):** TOKEN → SETUP (clone ด้วย token) → smoke → เมทริกซ์ (~2.5–3.5 ชม. ในเซสชัน 12 ชม. สบาย) → pytest → render → บันทึกผล (push กลับ branch เดิม)

**ของอยู่ไหน:** ทุกอย่างอยู่ใน `/kaggle/working` = แท็บ **Output** ของ notebook (กด Save Version เพื่อเก็บถาวร) และระหว่างเทรน push ขึ้น GitHub ทุก 100 steps ตามกติกา repo — เซสชันหลุดไว ก็รัน TOKEN → SETUP → เมทริกซ์ซ้ำแล้วต่อได้เลย

In [ ]:
# ============================================================
# TOKEN — รันเซลล์นี้ "ก่อนเซลล์อื่นทั้งหมด"
# repo เป็น private → ต้องใช้ token ตั้งแต่ขั้น clone (ไม่ใช่แค่ตอน push)
# ให้ token 1 ใน 3 วิธี:
#   1) วาง PAT ลงตัวแปร TOKEN ด้านล่าง (ค่าอยู่แค่ในเซสชันนี้ — อย่า save notebook ที่มี token ขึ้น GitHub)
#   2) Kaggle Secrets: Add-ons → Secrets → label: GITHUB_TOKEN → Attach (แนะนำ)
#   3) ปล่อยว่าง → เซลล์นี้จะถามแบบซ่อนอักษรตอนรัน
# ============================================================
TOKEN = ""   # ← วิธี 1: วาง PAT ตรงนี้ เช่น "github_pat_xxx" หรือ "ghp_xxx"

import getpass
from urllib.parse import quote

REPO_HOST_PATH = "github.com/gunnsmart/cot-hybrid-.git"   # ใช้ทั้ง clone และ push
SRC = None
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        try:
            TOKEN = (UserSecretsClient().get_secret("GITHUB_TOKEN") or "").strip()
            if TOKEN:
                SRC = "Kaggle Secrets (GITHUB_TOKEN)"
        except Exception:
            print("(ไม่มี secret GITHUB_TOKEN หรือยังไม่ได้กด Attach → Add-ons → Secrets → ลองถามตอนรันแทน)")
    except ImportError:
        pass
if not TOKEN:
    try:
        TOKEN = getpass.getpass("GitHub PAT (จำเป็น — repo เป็น private): ").strip()
        if TOKEN:
            SRC = "ป้อนเองตอนรัน"
    except Exception:
        TOKEN = ""

if not TOKEN:
    raise RuntimeError(
        "ไม่มี GitHub token — repo นี้เป็น private จึง clone ไม่ได้เลย\n"
        "→ วาง PAT ในตัวแปร TOKEN ด้านบน หรือเพิ่ม Kaggle Secrets label GITHUB_TOKEN แล้ว Attach แล้วรันเซลล์นี้ใหม่")

URL  = f"https://{quote(TOKEN, safe='')}@{REPO_HOST_PATH}"
PUSH = True
print(f"TOKEN OK (จาก {SRC or 'พิมพ์ในเซลล์'}) — ใช้ token นี้ทั้งตอน clone และ push (ไม่มีพิมพ์ค่าออกจอ)")

In [ ]:
# ============================================================
# SETUP — เช็ค internet → clone (ด้วย token) → deps → ตารางสถานะ  |  รันหลังเซลล์ TOKEN
# ============================================================
import json, os, socket, subprocess, sys

BRANCH   = "arena/01a0ede9-cot-hybrid"
BASE     = os.environ.get("WORK_BASE", "/kaggle/working")   # = แท็บ Output ของ notebook
REPO_DIR = os.path.join(BASE, "cot-hybrid-")
WORK_DIR = os.path.join(REPO_DIR, "interleaved-reasoning")
PY       = sys.executable
os.makedirs(BASE, exist_ok=True)

if "URL" not in globals():
    raise RuntimeError("รันเซลล์ TOKEN (เซลล์บนสุดของโค้ด) ก่อน — เซลล์นี้ใช้ token จากมันตอน clone")

# ---- preflight: internet เปิดหรือยัง (Kaggle ปิดเป็นค่าเริ่มต้น!)
try:
    socket.create_connection(("github.com", 443), timeout=8).close()
except Exception as e:
    raise RuntimeError(
        "เข้า internet ไม่ได้ — ต้องเปิดก่อน: Settings (แถบขวา) → Internet → Turn on\n"
        "(Kaggle ต้องยืนยันเบอร์โทรศัพท์ก่อนจะเปิด internet ได้)") from e
print("internet OK")

# ---- clone ด้วย token (ครั้งแรก) หรืออัปเดตเป็น state ล่าสุด (ครั้งถัด ๆ ไป)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("clone ด้วย token →", REPO_DIR)
    r = subprocess.run(["git", "clone", "-b", BRANCH, URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "clone ไม่สำเร็จ — ตรวจว่า PAT ถูกต้องและมีสิทธิ์ repo นี้\n"
            "(fine-grained: Contents Read/Write | classic: scope repo)\n"
            + (r.stderr or "").strip()[-400:])
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], capture_output=True, text=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"],
                       capture_output=True, text=True)
    print("อัปเดต repo:", (r.stdout or r.stderr).strip() or "up to date")

# ---- ยืนยันสิทธิ์ + branch มีจริง
v = subprocess.run(["git", "-C", REPO_DIR, "ls-remote", "origin", f"refs/heads/{BRANCH}"],
                   capture_output=True, text=True)
if v.returncode != 0 or not v.stdout.strip():
    raise RuntimeError("เข้าถึง branch ไม่ได้ — token ไม่มีสิทธิ์ repo นี้ หรือ branch ถูกลบ\n" + (v.stderr or "")[-300:])
print(f"เข้าถึง repo OK (branch {BRANCH}) — push ภายหลังใช้ token เดียวกัน")

for cfg in (("user.name", "kaggle-runner"), ("user.email", "kaggle-runner@users.noreply.github.com")):
    subprocess.run(["git", "-C", REPO_DIR, "config", *cfg], check=False)

subprocess.run([PY, "-m", "pip", "install", "-q", "pytest"], check=False)

def run_status(rid):
    p = os.path.join(WORK_DIR, "results", rid + ".json")
    if not os.path.exists(p):
        return "-"
    try:
        return json.load(open(p)).get("status", "?")
    except Exception:
        return "?"

ALL_RUNS = (["A_mlp_main", "B_mlp_main", "C_mlp_main"]
            + [f"{t}_{f}_base" for t in "ABC" for f in ("emit", "latent")]
            + ["A_abl_noprice", "A_abl_nocommit", "A_abl_nondeg", "A_abl_bare",
               "A_abl_sig0", "A_abl_reset", "A_abl_redundant", "A_abl_nogate"]
            + [f"A_scale_n{n}" for n in (4, 8, 12, 24, 48)]
            + ["A_gru", "A_ssm", "A_transformer",
               "A_gru_emit", "A_gru_latent", "A_ssm_emit", "A_ssm_latent",
               "A_tf_emit", "A_tf_latent", "B_gru", "B_transformer"])

os.chdir(WORK_DIR)
done = sum(run_status(r) == "done" for r in ALL_RUNS)
print(f"\nสถานะ: {done}/{len(ALL_RUNS)} runs เสร็จแล้ว")
for r in ALL_RUNS:
    print(f"  {r:18s} {run_status(r)}")
print("\nWORK_DIR =", WORK_DIR, " (โฟลเดอร์นี้คือแท็บ Output — กด Save Version เพื่อเก็บถาวร)")
print("\nSETUP OK — รันเซลล์ถัดไปได้")

In [ ]:
# ============================================================
# ทดสอบระบบ — เทรนจิ๋ว 30 steps (~<1 นาที) ให้มั่นใจว่าพร้อมก่อนรันจริง
# ============================================================
import os, shutil, subprocess
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")

r = subprocess.run([PY, "-m", "scripts.train", "--run-id", "_smoke", "--task", "recall",
                    "--arch", "mlp", "--d", "64", "--n", "4", "--depth-max", "3",
                    "--p-trivial", "0.0", "--steps", "30", "--eval-every", "30",
                    "--batch", "64", "--train-n", "512", "--no-git"],
                   capture_output=True, text=True)
print(r.stdout[-1200:])
if r.returncode != 0:
    print(r.stderr[-1500:])
    raise RuntimeError("smoke test ล้มเหลว — ดู error ด้านบน (อย่ารันเมทริกซ์ต่อจนแก้ได้)")
for junk in ("results/_smoke.json", "checkpoints/_smoke.pt"):
    if os.path.exists(junk):
        os.remove(junk)
shutil.rmtree("runs/_smoke", ignore_errors=True)
print("SMOKE OK — พร้อมรันเมทริกซ์ (ไปเซลล์ถัดไป)")

## รันเมทริกซ์ที่เหลือทั้งหมด (~2.5–3.5 ชม.)

สูตรเดียวกับ `scripts/run_matrix.sh` (recipe จาก calibration `A_cal27`: σ=0.1, gate q=0.30, expectation content):

1. `C_mlp_main` — **resume จาก step 500** (โหลด snapshot + นับ step จาก JSON ให้เอง)
2. baselines 6 รัน (`--force emit`/`--force latent` ครบ 3 tasks) — ใช้ตัดเกณฑ์ 4
3. ablations 8 รัน — พิสูจน์ 4 failure modes
4. scaling N = 4/8/12/24/48
5. generalization GRU / SSM / transformer (+ per-arch baselines, cross-task)

**ทนต่อการหลุด:** ระหว่างเทรน push ทุก 100 steps ตามกติกา repo — หลุดกลางทาง → รัน TOKEN + SETUP + เซลล์นี้ซ้ำ (อัน done ถูกข้าม, อันค้าง resume)
**อยากดูแผนโดยไม่เทรนจริง:** รัน `os.environ["MATRIX_DRY"] = "1"` ในเซลล์ใหม่ก่อน แล้วรันเซลล์นี้

In [ ]:
# ============================================================
# เมทริกซ์ไดรเวอร์ — ข้ามอัน done, resume อันค้าง (ทุก run เสร็จ = push ขึ้น GitHub แล้ว)
# ============================================================
import json, os, shutil, subprocess, time
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")

REC = ["--sigma", "0.1", "--lam-price", "0.2", "--lam-commit", "0.1", "--lam-nd", "20",
       "--reader-layers", "2", "--batch", "96", "--q-conf", "0.30", "--content",
       "expectation", "--push-every", "100"]

def A(steps, *extra):
    return ["--task", "arithmetic", "--arch", "mlp", "--d", "128", "--n", "12",
            "--depth-max", "4", "--p-trivial", "0.0", "--steps", str(steps), *REC, *extra]
def L(steps, *extra):
    return ["--task", "logic", "--arch", "mlp", "--d", "128", "--n", "12",
            "--depth-max", "5", "--steps", str(steps), *REC, *extra]
def R(steps, *extra):
    return ["--task", "recall", "--arch", "mlp", "--d", "128", "--n", "12",
            "--depth-max", "5", "--p-trivial", "0.0", "--steps", str(steps), *REC, *extra]
def G(arch, task="arithmetic", *extra):
    cmd = ["--task", task, "--arch", arch, "--d", "96", "--n", "12",
           "--depth-max", "4" if task == "arithmetic" else "5"]
    if task != "logic":
        cmd += ["--p-trivial", "0.0"]
    return cmd + ["--steps", "2000", *REC, *extra]
def S(n):
    return ["--task", "arithmetic", "--arch", "mlp", "--d", "96", "--n", str(n),
            "--depth-max", "4", "--p-trivial", "0.0", "--steps", "2000", *REC]

MATRIX = {
    "C_mlp_main":    R(3000),
    "A_emit_base":   A(3000, "--force", "emit"),
    "A_latent_base": A(3000, "--force", "latent"),
    "B_emit_base":   L(3000, "--force", "emit"),
    "B_latent_base": L(3000, "--force", "latent"),
    "C_emit_base":   R(3000, "--force", "emit"),
    "C_latent_base": R(3000, "--force", "latent"),
    "A_abl_noprice":   A(2500, "--lam-price", "0.0"),
    "A_abl_nocommit":  A(2500, "--lam-commit", "0.0"),
    "A_abl_nondeg":    A(2500, "--lam-nd", "0.0"),
    "A_abl_bare":      A(2500, "--lam-price", "0.0", "--lam-commit", "0.0", "--lam-nd", "0.0"),
    "A_abl_sig0":      A(2500, "--sigma", "0.0"),
    "A_abl_reset":     A(2500, "--semantics", "reset"),
    "A_abl_redundant": A(2500, "--p-trivial", "0.4"),
    "A_abl_nogate":    A(2500, "--q-conf", "0.0"),
    "A_scale_n4":  S(4),
    "A_scale_n8":  S(8),
    "A_scale_n12": S(12),
    "A_scale_n24": S(24),
    "A_scale_n48": S(48),
    "A_gru":         G("gru"),
    "A_ssm":         G("ssm"),
    "A_transformer": G("transformer"),
    "A_gru_emit":    G("gru", "arithmetic", "--force", "emit"),
    "A_gru_latent":  G("gru", "arithmetic", "--force", "latent"),
    "A_ssm_emit":    G("ssm", "arithmetic", "--force", "emit"),
    "A_ssm_latent":  G("ssm", "arithmetic", "--force", "latent"),
    "A_tf_emit":     G("transformer", "arithmetic", "--force", "emit"),
    "A_tf_latent":   G("transformer", "arithmetic", "--force", "latent"),
    "B_gru":         G("gru", "logic"),
    "B_transformer": G("transformer", "logic"),
}

def status(rid):
    p = os.path.join("results", rid + ".json")
    if os.path.exists(p):
        try:
            return json.load(open(p)).get("status")
        except Exception:
            return None
    return None

DRY = os.environ.get("MATRIX_DRY") == "1"
if DRY:
    print("DRY RUN — แสดงแผนเท่านั้น ไม่เทรนจริง\n")

T0, failed = time.time(), []
for rid, args in MATRIX.items():
    st = status(rid)
    if st == "done":
        print(f"SKIP   {rid} (done)")
        continue
    latest = os.path.join("runs", rid, "latest.pt")
    best = os.path.join("checkpoints", rid + ".pt")
    would_resume = st == "running" and (os.path.exists(latest) or os.path.exists(best))
    resume = []
    if st == "running" and not DRY:
        if not os.path.exists(latest) and os.path.exists(best):
            os.makedirs(os.path.join("runs", rid), exist_ok=True)
            shutil.copy(best, latest)
            print(f"SEED   {rid}: ทำ snapshot จาก best checkpoint เพื่อ resume")
        if os.path.exists(latest):
            resume = ["--resume"]
    if DRY:
        print(f"DRY    {rid}: {'resume ต่อจาก snapshot' if would_resume else 'รันใหม่'}")
        continue

    print(f"\n=== {rid} === [{(time.time()-T0)/60:.0f}m elapsed]", flush=True)
    cmd = [PY, "-m", "scripts.train", "--run-id", rid, *args, *resume]
    if not globals().get("PUSH", False):
        cmd.append("--no-git")
    r = subprocess.run(cmd)
    if r.returncode != 0:
        print(f"FAILED {rid} (exit {r.returncode})")
        failed.append(rid)
        if r.returncode == 2:
            print("push ล้มเหลว → หยุดตามกติกา repo: ตรวจ PAT/สิทธิ์ Contents:RW แล้วรัน TOKEN + SETUP + เซลล์นี้ใหม่ (อัน done จะถูกข้าม)")
            break
        continue

print(f"\nจบ: {len(MATRIX)-len(failed)}/{len(MATRIX)} ok | failed: {failed or 'ไม่มี'} | {(time.time()-T0)/60:.0f} นาที")
if not DRY:
    print("ขั้นถัดไป: รันเซลล์ pytest → render FINDINGS → บันทึกผล")

In [ ]:
# ============================================================
# ชุดทดสอบ: 6 เกณฑ์ + negative results + ความสม่ำเสมอของตัวเลข
# (suite เขียวได้ก็ต่อเมื่อเมทริกซ์ข้างบนครบ — ถ้ายังไม่ครบ ดูรายชื่อที่ fail ได้จาก output)
# ============================================================
import os, subprocess
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")
r = subprocess.run([PY, "-m", "pytest", "-q"])
if r.returncode != 0:
    print("\npytest ยังไม่เขียว — ถ้าเมทริกซ์ยังไม่ครบให้รันเซลล์เมทริกซ์ต่อก่อน แล้วกลับมารันเซลล์นี้")

In [ ]:
# ============================================================
# render FINDINGS.md (+ บล็อกผลใน README.md) จาก results/*.json
# ============================================================
import os, subprocess
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")
r = subprocess.run([PY, "-m", "scripts.render_findings"], capture_output=True, text=True)
print(r.stdout or "", r.stderr or "")
assert r.returncode == 0, "render_findings ล้มเหลว"
print(open("FINDINGS.md").read()[:2000])

In [ ]:
# ============================================================
# บันทึกผล — commit + push กลับ branch เดิม (token เดียวกับตอน clone)
#   ไฟล์ทั้งหมดอยู่ใน /kaggle/working = แท็บ Output อยู่แล้ว
#   → กด "Save Version" (Save & Run All หรือ Quick Save) เพื่อล็อกเป็น output ถาวร + โหลดกลับมาได้
# ============================================================
import os, subprocess, zipfile
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")
if "PUSH" not in globals():
    raise RuntimeError("รันเซลล์ TOKEN ก่อน (เซลล์นี้ใช้ token จากมันตอน push)")

subprocess.run(["git", "add", "-A"], check=True)
subprocess.run(["git", "commit", "-m", "kaggle: matrix complete + FINDINGS rendered", "--allow-empty"], check=False)
r = subprocess.run(["git", "push", "origin", BRANCH], capture_output=True, text=True)
print((r.stdout or r.stderr).strip()[-500:])
if r.returncode != 0:
    print("push ไม่สำเร็จ (non-fast-forward? → รัน SETUP ใหม่เพื่อ ff merge แล้ว push ซ้ำ)")
else:
    print("PUSH OK — ผลทั้งหมดอยู่บน GitHub แล้ว")

zip_path = os.path.join(BASE, "cot-hybrid-results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in ("results", "checkpoints", "FINDINGS.md", "README.md"):
        p = os.path.join(WORK_DIR, item)
        if os.path.isfile(p):
            z.write(p)
        elif os.path.isdir(p):
            for root, _, fns in os.walk(p):
                for fn in fns:
                    z.write(os.path.join(root, fn))
print("zip สำรองอยู่ใน Output:", zip_path)

## ปัญหาที่พบบ่อย (Kaggle)

| อาการ | สาเหตุ/วิธีแก้ |
|---|---|
| `เข้า internet ไม่ได้` | Settings (แถบขวา) → Internet → Turn on (ต้องยืนยันเบอร์โทรไว้กับ Kaggle ก่อน) |
| เซลล์ TOKEN ขึ้น `ไม่มี GitHub token` | วาง PAT ในตัวแปร `TOKEN` หรือเพิ่ม Secret label `GITHUB_TOKEN` + กด **Attach** แล้วรันเซลล์ใหม่ |
| `clone ไม่สำเร็จ` | PAT ผิด/หมดอายุ หรือไม่มีสิทธิ์ repo (fine-grained: Contents Read/**Write** | classic: scope `repo`) |
| `RuntimeError: รันเซลล์ ... ก่อน` | runtime รีเซ็ต/รันข้ามลำดับ → ไล่ TOKEN → SETUP → เซลล์ถัดไป (อัน done ไม่หาย เพราะ push ไว้ทุก 100 steps) |
| เซสชันหลุดกลางเมทริกซ์ (CPU ได้ถึง 12 ชม./เซสชัน) | เปิด notebook ใหม่ → TOKEN → SETUP → เซลล์เมทริกซ์ ต่อทันที อัน done ถูกข้าม อันค้าง resume |
| push fail (exit 2) ตอนเทรน | PAT ไม่มีสิทธิ์เขียน → สร้าง PAT ใหม่ Contents: RW แล้วไล่ TOKEN + SETUP + เมทริกซ์ซ้ำ |
| push บอก non-fast-forward | มี push จากที่อื่นทับหน้า → รัน SETUP (merge --ff-only ให้) แล้วกดเซลล์บันทึกผลซ้ำ |
| เอาไฟล์ผลออกจาก Kaggle | แท็บ **Output** ขวามือ (หรือกด Save Version ให้เป็น output ถาวร) — zip รวมอยู่ใน `cot-hybrid-results.zip` |
| อยากรู้ก่อนว่าจะรันอะไรบ้าง | `os.environ["MATRIX_DRY"]="1"` แล้วรันเซลล์เมทริกซ์ = แสดงแผนอย่างเดียว |
| ทำไมไม่ใช้ GPU | โค้ดล็อก `DEVICE="cpu"` เพื่อ reproducibility (noise ใช้ seeded generator) — Kaggle CPU 4 คอร์ฟรีไม่จำกัดเวลาสัปดาห์ เหมาะกว่าเผา GPU quota |

**ความปลอดภัยของ token:** แนะนำ Kaggle Secrets (ไม่ลงโค้ด) / ถ้าพิมพ์ในเซลล์ **อย่า save notebook ตัวนี้ทับขึ้น GitHub** / token ไม่เคยถูก print ออกจอ / ควร revoke เมื่อใช้เสร็จ